Cell 1 — Imports & Helper Functions

In [1]:
# ============================================================
# HOUSE PRICE PREDICTION NOTEBOOK
# ============================================================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from joblib import dump

# Helper function for validated numeric input with min/max ranges
def get_validated_float(prompt, min_val=None, max_val=None):
    while True:
        user_input = input(prompt + ": ").strip().lower()
        try:
            value = float(user_input)
            if min_val is not None and value < min_val:
                print(f"  Error: Value must be at least {min_val}. (You entered: {value})")
                continue
            if max_val is not None and value > max_val:
                print(f"  Error: Value must be at most {max_val}. (You entered: {value})")
                continue
            return value
        except ValueError:
            print("  Please enter a valid number.")


Cell 2 — Load Dataset from Local CSVs (Robust Paths)

In [2]:
# ------------------------------------------------------------
# Robust path handling for CSVs
# ------------------------------------------------------------
notebook_dir = os.getcwd()  # current working directory

train_csv_path = os.path.abspath(os.path.join(notebook_dir, "..", "data", "raw", "california_housing_train.csv"))
test_csv_path  = os.path.abspath(os.path.join(notebook_dir, "..", "data", "raw", "california_housing_test.csv"))

print(f"Looking for train CSV at: {train_csv_path}")
print(f"Looking for test CSV at:  {test_csv_path}")

# Load CSVs and create derived features
try:
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)

    full_df = pd.concat([train_df, test_df], ignore_index=True)

    # Derived features
    full_df['AveRooms'] = full_df['total_rooms'] / full_df['households'].replace(0, np.nan)
    full_df['AveBedrms'] = full_df['total_bedrooms'] / full_df['households'].replace(0, np.nan)
    full_df['AveOccup'] = full_df['population'] / full_df['households'].replace(0, np.nan)

    # Features and target
    X = full_df[['longitude', 'latitude', 'housing_median_age', 'population', 'median_income',
                 'AveRooms', 'AveBedrms', 'AveOccup']]
    y = full_df['median_house_value']

    # Fill NaNs
    X = X.fillna(X.mean())

    print("Dataset loaded and standardized successfully!")

except FileNotFoundError:
    print(f"Error: Could not find CSV files at {train_csv_path} and {test_csv_path}.")
    raise
except Exception as e:
    print(f"Unexpected error loading data: {e}")
    raise


Looking for train CSV at: C:\Users\charo\ai-portfolio\data\raw\california_housing_train.csv
Looking for test CSV at:  C:\Users\charo\ai-portfolio\data\raw\california_housing_test.csv
Dataset loaded and standardized successfully!


In [ ]:
Cell 3 — Train/Test Split & Model Training

In [3]:
# ------------------------------------------------------------
# Train/Test Split and Random Forest Model
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# Optional: filter extreme house prices to reduce outliers
full_df = full_df[(full_df['median_house_value'] >= 50000) & 
                  (full_df['median_house_value'] <= 500000)]

# Features and target
X = full_df[['longitude', 'latitude', 'housing_median_age', 'population', 'median_income',
             'AveRooms', 'AveBedrms', 'AveOccup']]
y = full_df['median_house_value']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Optional: log-transform target to reduce effect of outliers
# y_train_log = np.log1p(y_train)
# y_test_log = np.log1p(y_test)

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)  # Use y_train_log here if you want log-transform

print("✅ Random Forest model trained successfully!")


✅ Random Forest model trained successfully!


Cell 4 – Evaluate Mode

In [4]:
# ------------------------------------------------------------
# Evaluate Random Forest Model
# ------------------------------------------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict on test set
y_pred_test = model.predict(X_test)  # Use np.expm1(y_pred_test) if log-transform was used

# Optional: clip predictions to realistic bounds
y_pred_test = np.clip(y_pred_test, 50000, 500000)

# Display metrics
print("\n=== Test Set Metrics ===")
print("MAE:", round(mean_absolute_error(y_test, y_pred_test), 3))
print("MSE:", round(mean_squared_error(y_test, y_pred_test), 3))
print("R² Score:", round(r2_score(y_test, y_pred_test), 3))



=== Test Set Metrics ===
MAE: 30921.95
MSE: 2123466152.393
R² Score: 0.779


Cell 5 – Save Trained Model

In [5]:
# ------------------------------------------------------------
# Save Trained Model
# ------------------------------------------------------------
import os
from joblib import dump

# Ensure 'models' directory exists
model_dir = os.path.abspath(os.path.join(notebook_dir, "..", "models"))
os.makedirs(model_dir, exist_ok=True)

# Path for the .pk1 file
model_path = os.path.join(model_dir, "California_house_value_model.pk1")

# Save the model
dump(model, model_path)
print(f"✅ Model saved to: {model_path}")


✅ Model saved to: C:\Users\charo\ai-portfolio\models\California_house_value_model.pk1


In [ ]:
Cell 6 - Create zipcode file 

In [27]:
import os
import pandas as pd

# -----------------------------
# Define ZIP ranges and city info
# -----------------------------
data = [
    # Southern California
    {"start":90001, "end":90089, "city":"Los Angeles", "state":"CA", "region":"Southern California", "lat":34.05, "lon":-118.25},
    {"start":90210, "end":90210, "city":"Beverly Hills", "state":"CA", "region":"Southern California", "lat":34.07, "lon":-118.41},
    {"start":92101, "end":92199, "city":"San Diego", "state":"CA", "region":"Southern California", "lat":32.72, "lon":-117.16},
    {"start":92501, "end":92501, "city":"Riverside", "state":"CA", "region":"Southern California", "lat":33.98, "lon":-117.37},
    {"start":92801, "end":92801, "city":"Anaheim", "state":"CA", "region":"Southern California", "lat":33.84, "lon":-117.91},
    {"start":93101, "end":93101, "city":"Santa Barbara", "state":"CA", "region":"Southern California", "lat":34.42, "lon":-119.70},
    
    # Central California
    {"start":93650, "end":93650, "city":"Fresno area", "state":"CA", "region":"Central California", "lat":36.74, "lon":-119.78},
    {"start":93701, "end":93728, "city":"Fresno", "state":"CA", "region":"Central California", "lat":36.74, "lon":-119.78},
    {"start":93901, "end":93901, "city":"Salinas", "state":"CA", "region":"Central California", "lat":36.67, "lon":-121.65},
    {"start":95014, "end":95014, "city":"Cupertino", "state":"CA", "region":"Central California", "lat":37.32, "lon":-122.03},
    {"start":95110, "end":95136, "city":"San Jose", "state":"CA", "region":"Central California", "lat":37.33, "lon":-121.89},
    
    # Northern California
    {"start":94101, "end":94188, "city":"San Francisco", "state":"CA", "region":"Northern California", "lat":37.77, "lon":-122.42},
    {"start":94016, "end":94016, "city":"Daly City", "state":"CA", "region":"Northern California", "lat":37.69, "lon":-122.47},
    {"start":94301, "end":94301, "city":"Palo Alto", "state":"CA", "region":"Northern California", "lat":37.44, "lon":-122.16},
    {"start":94701, "end":94701, "city":"Berkeley", "state":"CA", "region":"Northern California", "lat":37.87, "lon":-122.27},
    {"start":94501, "end":94501, "city":"Alameda", "state":"CA", "region":"Northern California", "lat":37.77, "lon":-122.25},
    {"start":95814, "end":95814, "city":"Sacramento", "state":"CA", "region":"Northern California", "lat":38.58, "lon":-121.49},
    {"start":95630, "end":95630, "city":"Folsom", "state":"CA", "region":"Northern California", "lat":38.68, "lon":-121.16},
    
    # Far North / Border
    {"start":95501, "end":95501, "city":"Eureka", "state":"CA", "region":"Far North / Border", "lat":40.80, "lon":-124.16},
    {"start":96001, "end":96001, "city":"Redding", "state":"CA", "region":"Far North / Border", "lat":40.58, "lon":-122.38}
]

# -----------------------------
# Expand ZIP ranges
# -----------------------------
rows = []
for entry in data:
    for zip_code in range(entry["start"], entry["end"] + 1):
        rows.append({
            "zip": zip_code,
            "city": entry["city"],
            "state": entry["state"],
            "region": entry["region"],
            "latitude": entry["lat"],
            "longitude": entry["lon"]
        })
# -----------------------------
# Create DataFrame and save CSV
# -----------------------------
df = pd.DataFrame(rows)

# Ensure output directory exists
raw_data_dir = os.path.abspath(os.path.join(notebook_dir,"..","data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

# Full path for CSV
raw_data_path = os.path.join(raw_data_dir, "california_zip_lat_long_full.csv")

# Save CSV
df.to_csv(raw_data_path, index=False)
print(f"✅ CSV saved  {raw_data_path}")

✅ CSV saved  C:\Users\charo\ai-portfolio\data\raw\california_zip_lat_long_full.csv


In [28]:
import pandas as pd
import os

# Load the California zipcode data
zipcode_csv_path = os.path.abspath(os.path.join(notebook_dir, "..", "data", "raw", "california_zip_lat_long_full.csv"))

try:
    zipcode_df = pd.read_csv(zipcode_csv_path)
    zipcode_df.columns = zipcode_df.columns.str.strip().str.upper() # Standardize column names
    # Assuming columns are like 'ZIP', 'LATITUDE', 'LONGITUDE'
    # Let's verify column names first and rename if necessary
    if 'ZIP' in zipcode_df.columns and 'LATITUDE' in zipcode_df.columns and 'LONGITUDE' in zipcode_df.columns:
        # Rename 'LATITUDE' to 'LAT' and 'LONGITUDE' to 'LONG' to match expected names for consistency if needed later
        # For now, we'll ensure they are present and use them directly.
        print("✅ Zipcode data loaded successfully!")
        # Display first few rows and info for verification
        print(zipcode_df.head())
    else:
        # If the columns are not exactly LATITUDE and LONGITUDE, check for LAT and LON/LONG as fallback
        if 'ZIP' in zipcode_df.columns and 'LAT' in zipcode_df.columns and ('LON' in zipcode_df.columns or 'LONG' in zipcode_df.columns):
            if 'LON' in zipcode_df.columns:
                zipcode_df.rename(columns={'LON': 'LONG'}, inplace=True)
            print("✅ Zipcode data loaded successfully (with LAT/LONG alias)!")
            print(zipcode_df.head())
        else:
            raise ValueError("Expected columns 'ZIP', 'LATITUDE', 'LONGITUDE' or ('ZIP', 'LAT', and ('LON' or 'LONG')) not found in the zipcode file.")

except FileNotFoundError:
    print(f"Error: Zipcode CSV file not found at {zipcode_csv_path}. Please ensure it's in the correct directory.")
    raise
except Exception as e:
    print(f"Unexpected error loading zipcode data: {e}")
    raise


✅ Zipcode data loaded successfully!
     ZIP         CITY STATE               REGION  LATITUDE  LONGITUDE
0  90001  Los Angeles    CA  Southern California     34.05    -118.25
1  90002  Los Angeles    CA  Southern California     34.05    -118.25
2  90003  Los Angeles    CA  Southern California     34.05    -118.25
3  90004  Los Angeles    CA  Southern California     34.05    -118.25
4  90005  Los Angeles    CA  Southern California     34.05    -118.25


In [ ]:
Cell 6 — Interactive Realistic Prediction test user inputs

In [29]:
### ------------------------------------------------------------
# Interactive prediction with realistic input ranges
# ------------------------------------------------------------
print("\n--- Realistic House Price Prediction ---")

while True:
    median_income = get_validated_float("Median Income (10k USD, 0.5–15)", 0.1, 15.0)
    housing_median_age = get_validated_float("House Age (years, 1–60)", 1.0, 60.0)
    AveRooms = get_validated_float("Average Rooms per Household (1–15)", 1.0, 15.0)
    AveBedrms = get_validated_float("Average Bedrooms (0.1–5)", 0.1, 5.0)
    population = get_validated_float("Population (1–10,000)", 1.0, 10000.0)
    AveOccup = get_validated_float("Average Occupancy (0.1–20)", 0.1, 20.0)

    # Get zipcode and look up latitude/longitude
    while True:
        zipcode_input = input("Enter Zipcode: ").strip()
        if not zipcode_input.isdigit():
            print("  Error: Zipcode must be a number.")
            continue
        zipcode_input = int(zipcode_input)
        zip_info = zipcode_df[zipcode_df['ZIP'] == zipcode_input]

        if not zip_info.empty:
            latitude = zip_info['LATITUDE'].iloc[0]
            longitude = zip_info['LONGITUDE'].iloc[0]
            print(f"  Found Lat: {latitude}, Long: {longitude} for Zipcode {zipcode_input}")
            break # Exit the zipcode input loop
        else:
            print(f"  Error: Zipcode {zipcode_input} not found in our database. Please try again.")
            # The loop will continue, prompting for another zipcode

    new_house = pd.DataFrame([[
        longitude,
        latitude,
        housing_median_age,
        population,
        median_income,
        AveRooms,
        AveBedrms,
        AveOccup
    ]], columns=X.columns) # Changed X_train.columns to X.columns

    predicted_price = model.predict(new_house)[0]
    print(f"\nPredicted Median House Value: ${predicted_price:,.2f}")

    if predicted_price < 10000 or predicted_price > 5000000:
        print("*** Predicted value seems UNREALISTIC. ***")
        retry = input("Re-enter ALL values? (y-yes / q-quit): ").strip().lower()
        if retry in ['q', 'quit']:
            break
    else:
        print("Predicted value is within a realistic range.")
        break



--- Realistic House Price Prediction ---


Median Income (10k USD, 0.5–15):  5
House Age (years, 1–60):  10
Average Rooms per Household (1–15):  8
Average Bedrooms (0.1–5):  3
Population (1–10,000):  2000
Average Occupancy (0.1–20):  3
Enter Zipcode:  95630


  Found Lat: 38.68, Long: -121.16 for Zipcode 95630

Predicted Median House Value: $271,356.55
Predicted value is within a realistic range.
